### Dependencies

In [1]:
# Cell 0 — ONLY if the runtime has just started
!pip install -q huggingface_hub==0.23.0 transformers==4.39.3 diffusers==0.27.2 peft==0.10.0
!pip install -q insightface onnxruntime-gpu onnx onnx2torch
!pip install -q git+https://github.com/tencent-ailab/IP-Adapter.git

print("✓ Libraries installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 122.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.0 which is incompatible.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.39.3 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.23.0 which is incompatib

## Common setup


In [ ]:
# === Common project bootstrap ===
# Bootstrap Colab/Drive + src path
import os, sys

DRIVE = "/content/drive/MyDrive/Notebook4"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

for _path in [DRIVE, "/content", "/content/IP-Adapter"]:
    if _path not in sys.path:
        sys.path.insert(0, _path)


from src.config import DEFAULT_DRIVE
from src.inference_utils import (
    CHILD_NEGATIVE_PROMPT,
    DEFAULT_NEGATIVE_PROMPT,
    collect_bap2_alpha_stats,
    ensure_2d,
    generate_bap2_image,
    get_embedding,
    set_parent_scale_bap2,
)

from src.notebook_setup import setup_runtime, setup_data, setup_generation_models, setup_arcface
from src.model_utils import install_bap2, processor_counts
DRIVE = globals().get("DRIVE", DEFAULT_DRIVE)
ctx = setup_runtime(DRIVE, require_cuda=True)
ctx = setup_data(ctx)
ctx = setup_generation_models(ctx)
ctx = setup_arcface(ctx)

paths = ctx.paths
runtime = ctx.runtime
data = ctx.data
pipe = ctx.pipe
ip_model = ctx.ip_model
arcface_torch = ctx.arcface_torch

DEVICE = runtime.device
DTYPE = runtime.dtype
FIW_ROOT = str(paths.fiw_root)
EMBEDDINGS_PATH = str(paths.embeddings_path)
TRAIN_CSV = str(paths.train_csv)
VAL_CSV = str(paths.val_csv)
TEST_CSV = str(paths.test_csv)
AGES_CSV = str(paths.ages_csv)
GENDER_CACHE = str(paths.gender_cache_path)

embeddings = data.embeddings
gender_dict = data.gender_dict
train_ds = data.train_ds
val_ds = data.val_ds
test_ds = data.test_ds
train_loader = data.train_loader
val_loader = data.val_loader
test_loader = data.test_loader

bap2_modules = install_bap2(pipe, runtime, parent_scale=0.1, copy_ip_weights=True)
START_STEP = globals().get('START_STEP', 0)
CURRENT_STEP = globals().get('CURRENT_STEP', START_STEP)

print('Training context ready')
print('Processors:', processor_counts(pipe.unet))
print('BAP2 modules:', len(bap2_modules))


# Helpers finales comunes
import torch
from src.inference_utils import make_arcface_embedder

GUIDANCE = globals().get("GUIDANCE", 4.0)
STEPS = globals().get("STEPS", 30)
NEG_PROMPT = globals().get("NEG_PROMPT", DEFAULT_NEGATIVE_PROMPT)
arcface_embed_from_pil = make_arcface_embedder(arcface_torch, DEVICE)

@torch.inference_mode()
def generate_bap2_final(h_c, h_p, h_m, gender, age, parent_scale, seed):
    img, prompt, alpha_stats = generate_bap2_image(
        pipe=pipe,
        ip_model=ip_model,
        h_c=h_c,
        h_p=h_p,
        h_m=h_m,
        gender=gender,
        age=age,
        device=DEVICE,
        dtype=DTYPE,
        parent_scale=parent_scale,
        seed=seed,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        negative_prompt=NEG_PROMPT,
        child_prompt=False,
    )
    return img, alpha_stats, prompt

print('Helpers finales listos')


In [9]:
# ============================================================
# Cell 8 — Continue BAP2 training up to step500
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime

TARGET_STEP = 500

if START_STEP >= TARGET_STEP:
    CURRENT_STEP = START_STEP
    print(f"START_STEP={START_STEP}. TARGET_STEP={TARGET_STEP} already reached. Not training further.")

else:
    BATCH_SIZE = 1
    LR_BAP2 = 1e-4
    WEIGHT_DECAY = 0.0
    ADAM_EPS = 1e-6
    MAX_GRAD_NORM = 1.0
    LOG_EVERY = 10
    SAVE_EVERY = 100

    RUN_NAME = "bap2_linear_continue_step100_to_500_lr1e-4"
    BAP2_CONTINUE_DIR = f"{DRIVE}/results/child_anchor_bap_gate/{RUN_NAME}"
    os.makedirs(BAP2_CONTINUE_DIR, exist_ok=True)

    print("Results in:", BAP2_CONTINUE_DIR)
    print("Continuing from step", START_STEP, "to step", TARGET_STEP)

    pipe.unet.train()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    for p in pipe.unet.parameters():
        p.requires_grad = False

    for p in pipe.vae.parameters():
        p.requires_grad = False

    for p in pipe.text_encoder.parameters():
        p.requires_grad = False

    for p in ip_model.image_proj_model.parameters():
        p.requires_grad = False

    bap2_gate_params = []

    for name, proc in bap2_modules.items():
        proc.gate.float()
        for p in proc.gate.parameters():
            p.requires_grad = True
            bap2_gate_params.append(p)

    print("BAP2 gate params entrenables:", sum(p.numel() for p in bap2_gate_params))
    print("dtype example:", bap2_gate_params[0].dtype)

    optimizer = torch.optim.AdamW(
        bap2_gate_params,
        lr=LR_BAP2,
        eps=ADAM_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    def ensure_2d(x):
        if x.ndim == 1:
            x = x.unsqueeze(0)
        return x

    def collect_bap2_alpha_stats():
        vals = []

        for name, proc in bap2_modules.items():
            if getattr(proc, "last_alpha", None) is not None:
                vals.append(proc.last_alpha.detach().float().cpu())

        if len(vals) == 0:
            return {
                "alpha_mean": None,
                "alpha_min": None,
                "alpha_max": None,
                "alpha_std": None,
                "alpha_has_nan": None,
            }

        x = torch.cat(vals, dim=0)

        return {
            "alpha_mean": x.mean().item(),
            "alpha_min": x.min().item(),
            "alpha_max": x.max().item(),
            "alpha_std": x.std().item(),
            "alpha_has_nan": torch.isnan(x).any().item(),
        }

    def bap2_param_stats():
        vals = []

        for name, proc in bap2_modules.items():
            for p in proc.gate.parameters():
                vals.append(p.detach().float().flatten().cpu())

        x = torch.cat(vals)

        return {
            "gate_param_mean": x.mean().item(),
            "gate_param_std": x.std().item(),
            "gate_param_max_abs": x.abs().max().item(),
            "gate_param_has_nan": torch.isnan(x).any().item(),
        }

    def build_prompts(batch):
        ages = batch["age"]
        genders = batch["gender"]

        if torch.is_tensor(ages):
            ages = ages.detach().cpu().tolist()

        if isinstance(genders, str):
            genders = [genders]

        prompts = []

        for gender, age in zip(genders, ages):
            if gender not in ["man", "woman", "person"]:
                gender = "person"

            prompts.append(
                f"portrait photo of a {gender}, age {int(age)}, realistic, neutral expression"
            )

        return prompts

    def train_step_bap2(batch):
        img = batch["img"].to(device=DEVICE, dtype=DTYPE)

        h_c = ensure_2d(batch["h_c"].to(device=DEVICE, dtype=DTYPE))
        h_p = ensure_2d(batch["h_p"].to(device=DEVICE, dtype=DTYPE))
        h_m = ensure_2d(batch["h_m"].to(device=DEVICE, dtype=DTYPE))

        prompts = build_prompts(batch)

        with torch.no_grad():
            latents = pipe.vae.encode(img).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor

            noise = torch.randn_like(latents)

            timesteps = torch.randint(
                low=0,
                high=pipe.scheduler.config.num_train_timesteps,
                size=(latents.shape[0],),
                device=DEVICE,
            ).long()

            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

            child_tokens = ip_model.image_proj_model(h_c)
            father_tokens = ip_model.image_proj_model(h_p)
            mother_tokens = ip_model.image_proj_model(h_m)

            prompt_embeds, _ = pipe.encode_prompt(
                prompt=prompts,
                device=DEVICE,
                num_images_per_prompt=1,
                do_classifier_free_guidance=False,
            )

            encoder_hidden_states = torch.cat(
                [prompt_embeds, child_tokens, father_tokens, mother_tokens],
                dim=1,
            )

        noise_pred = pipe.unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=encoder_hidden_states,
        ).sample

        loss = F.mse_loss(noise_pred.float(), noise.float())

        return loss

    def save_bap2_checkpoint(global_step, logs):
        bap2_gate_state = {
            name: proc.gate.state_dict()
            for name, proc in bap2_modules.items()
        }

        ckpt_path = f"{BAP2_CONTINUE_DIR}/bap2_linear_step{global_step}.pt"

        torch.save(
            {
                "bap2_gate_state": bap2_gate_state,
                "step": global_step,
                "config": {
                    "run_name": RUN_NAME,
                    "loss": "L_diff",
                    "start_step": START_STEP,
                    "target_step": TARGET_STEP,
                    "batch_size": BATCH_SIZE,
                    "lr_bap2": LR_BAP2,
                    "adam_eps": ADAM_EPS,
                    "weight_decay": WEIGHT_DECAY,
                    "max_grad_norm": MAX_GRAD_NORM,
                    "parent_scale": 0.1,
                    "child_scale": 1.0,
                    "gate": "contextual_linear",
                    "gate_dtype": "float32",
                    "timestamp": datetime.now().isoformat(),
                },
                "logs": logs,
            },
            ckpt_path,
        )

        return ckpt_path

    logs_continue = []
    global_step = START_STEP
    loader_iter = iter(train_loader)

    pbar = tqdm(total=TARGET_STEP - START_STEP)

    while global_step < TARGET_STEP:

        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch = next(loader_iter)

        batch = {
            k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        loss = train_step_bap2(batch)

        if not torch.isfinite(loss):
            print("⚠ Non-finite loss at global_step", global_step + 1, "loss:", loss.item())
            break

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            bap2_gate_params,
            max_norm=MAX_GRAD_NORM,
        )

        optimizer.step()

        global_step += 1

        alpha_stats = collect_bap2_alpha_stats()
        param_stats = bap2_param_stats()

        row = {
            "global_step": global_step,
            "loss": loss.item(),
            "grad_norm": float(grad_norm),
            **alpha_stats,
            **param_stats,
        }

        logs_continue.append(row)

        if global_step % LOG_EVERY == 0 or global_step == START_STEP + 1:
            print(
                f"step {global_step:04d} | "
                f"loss={loss.item():.6f} | "
                f"grad={float(grad_norm):.6e} | "
                f"alpha_mean={alpha_stats['alpha_mean']:.6f} | "
                f"alpha_std={alpha_stats['alpha_std']:.6f} | "
                f"alpha_min={alpha_stats['alpha_min']:.6f} | "
                f"alpha_max={alpha_stats['alpha_max']:.6f} | "
                f"param_max_abs={param_stats['gate_param_max_abs']:.6e}"
            )

        if alpha_stats["alpha_has_nan"] or param_stats["gate_param_has_nan"]:
            print("⚠ NaN detected in alpha or parameters. Stopping.")
            break

        if alpha_stats["alpha_min"] < 0.45 or alpha_stats["alpha_max"] > 0.55:
            print("⚠ Possible alpha collapse. Stopping.")
            print(alpha_stats)
            break

        if global_step % SAVE_EVERY == 0:
            ckpt_path = save_bap2_checkpoint(global_step, logs_continue)
            print(f"✓ Checkpoint saved: {ckpt_path}")

        pbar.update(1)

    pbar.close()

    df_bap2_continue = pd.DataFrame(logs_continue)
    display(df_bap2_continue.tail())

    logs_path = f"{BAP2_CONTINUE_DIR}/bap2_continue_logs_step{START_STEP}_to_{global_step}.csv"
    df_bap2_continue.to_csv(logs_path, index=False)

    final_ckpt_path = save_bap2_checkpoint(global_step, logs_continue)

    print("\n✓ Entrenamiento continuo terminado")
    print("Last global_step:", global_step)
    print("Logs:", logs_path)
    print("Checkpoint final:", final_ckpt_path)

    CURRENT_STEP = global_step
    START_STEP = global_step
    print("CURRENT_STEP:", CURRENT_STEP)

    if len(df_bap2_continue) > 0:
        print("\nAlpha summary:")
        print("alpha_mean inicial:", df_bap2_continue.iloc[0]["alpha_mean"])
        print("alpha_mean final:  ", df_bap2_continue.iloc[-1]["alpha_mean"])
        print("alpha_std inicial: ", df_bap2_continue.iloc[0]["alpha_std"])
        print("alpha_std final:   ", df_bap2_continue.iloc[-1]["alpha_std"])
        print("alpha_min final:   ", df_bap2_continue.iloc[-1]["alpha_min"])
        print("alpha_max final:   ", df_bap2_continue.iloc[-1]["alpha_max"])

        print("\nLoss summary:")
        print("loss inicial:", df_bap2_continue.iloc[0]["loss"])
        print("loss final:  ", df_bap2_continue.iloc[-1]["loss"])

Results in: /content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_linear_continue_step100_to_500_lr1e-4
Continuing from step 100 a step 500
BAP2 gate params entrenables: 792832
dtype example: torch.float32


  0%|          | 0/400 [00:00<?, ?it/s]

step 0101 | loss=0.022591 | grad=7.249935e-05 | alpha_mean=0.499809 | alpha_std=0.002612 | alpha_min=0.486679 | alpha_max=0.508824 | param_max_abs=2.065763e-03
step 0110 | loss=0.006452 | grad=5.240910e-06 | alpha_mean=0.499694 | alpha_std=0.003097 | alpha_min=0.484620 | alpha_max=0.511557 | param_max_abs=2.037936e-03
step 0120 | loss=0.026962 | grad=5.547014e-05 | alpha_mean=0.499652 | alpha_std=0.003263 | alpha_min=0.484679 | alpha_max=0.511789 | param_max_abs=2.076948e-03
step 0130 | loss=0.070036 | grad=1.971656e-04 | alpha_mean=0.499823 | alpha_std=0.002977 | alpha_min=0.485831 | alpha_max=0.509898 | param_max_abs=2.216047e-03
step 0140 | loss=0.008234 | grad=3.326096e-06 | alpha_mean=0.499794 | alpha_std=0.003218 | alpha_min=0.483975 | alpha_max=0.514205 | param_max_abs=2.292941e-03
step 0150 | loss=0.003940 | grad=0.000000e+00 | alpha_mean=0.499646 | alpha_std=0.004290 | alpha_min=0.477117 | alpha_max=0.516196 | param_max_abs=2.112384e-03
step 0160 | loss=0.177620 | grad=1.94884

,global_step,loss,grad_norm,alpha_mean,alpha_min,alpha_max,alpha_std,alpha_has_nan,gate_param_mean,gate_param_std,gate_param_max_abs,gate_param_has_nan
395,496,0.006353,0.000002,0.499821,0.473938,0.518903,0.005757,False,-7.029519e-08,0.000262,0.003912,False
396,497,0.015702,0.000037,0.499779,0.466503,0.518239,0.005834,False,-7.243565e-08,0.000263,0.003919,False
397,498,0.008543,0.000003,0.499812,0.479900,0.517648,0.005480,False,-7.427204e-08,0.000263,0.003926,False
398,499,0.020480,0.000031,0.499935,0.471456,0.522486,0.006544,False,-7.633989e-08,0.000263,0.003930,False
399,500,0.010698,0.000023,0.499751,0.469495,0.518771,0.006084,False,-7.876925e-08,0.000263,0.003933,False



✓ Entrenamiento continuo terminado
Last global_step: 500
Logs: /content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_linear_continue_step100_to_500_lr1e-4/bap2_continue_logs_step100_to_500.csv
Checkpoint final: /content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_linear_continue_step100_to_500_lr1e-4/bap2_linear_step500.pt
CURRENT_STEP: 500

Alpha summary:
alpha_mean inicial: 0.49980852007865906
alpha_mean final:   0.4997512102127075
alpha_std inicial:  0.0026117393281310797
alpha_std final:    0.006084279622882605
alpha_min final:    0.46949541568756104
alpha_max final:    0.5187708139419556

Loss summary:
loss inicial: 0.022590555250644684
loss final:   0.01069796271622181


In [10]:
# ============================================================
# Cell 9 — BAP2 evaluation: λ=0.0 vs λ=0.1
# ============================================================

import os
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import pandas as pd
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# ----------------------------
# Configuration
# ----------------------------
N_TRIADS = 10
AGES = [15, 25, 40, 60]
PARENT_SCALES = [0.0, 0.1]

SEED = 42
GUIDANCE = 4.0
STEPS = 30

negative_prompt = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"

# Fixed indices. Change them if you want.
EVAL_IDXS = [179, 12, 35, 48, 61, 74, 89, 103, 128, 150]
EVAL_IDXS = EVAL_IDXS[:N_TRIADS]

print("EVAL_IDXS:", EVAL_IDXS)

# ----------------------------
# ArcFace preprocessing
# ----------------------------
preprocess_arcface = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

@torch.inference_mode()
def arcface_embed_from_pil(img):
    x = preprocess_arcface(img.convert("RGB")).unsqueeze(0).to(DEVICE)
    x = x.float()
    emb = arcface_torch(x)
    emb = F.normalize(emb, p=2, dim=1)
    return emb

def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(unet, value):
    for proc in unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def collect_bap2_alpha_stats(unet):
    vals = []

    for name, proc in unet.attn_processors.items():
        if type(proc).__name__ == "BAP2":
            if getattr(proc, "last_alpha", None) is not None:
                vals.append(proc.last_alpha.detach().float().cpu())

    if len(vals) == 0:
        return None

    x = torch.cat(vals, dim=0)

    return {
        "alpha_mean": x.mean().item(),
        "alpha_min": x.min().item(),
        "alpha_max": x.max().item(),
        "alpha_std": x.std().item(),
    }

@torch.inference_mode()
def generate_bap2_age(h_c, h_p, h_m, gender, age, parent_scale, seed=42):
    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    set_parent_scale_bap2(pipe.unet, parent_scale)

    h_c = ensure_2d(h_c).to(device=DEVICE, dtype=DTYPE)
    h_p = ensure_2d(h_p).to(device=DEVICE, dtype=DTYPE)
    h_m = ensure_2d(h_m).to(device=DEVICE, dtype=DTYPE)

    child_tokens = ip_model.image_proj_model(h_c)
    father_tokens = ip_model.image_proj_model(h_p)
    mother_tokens = ip_model.image_proj_model(h_m)

    zero_child_tokens = ip_model.image_proj_model(torch.zeros_like(h_c))
    zero_father_tokens = ip_model.image_proj_model(torch.zeros_like(h_p))
    zero_mother_tokens = ip_model.image_proj_model(torch.zeros_like(h_m))

    prompt = f"portrait photo of a {gender}, age {age}, realistic, neutral expression"

    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=[prompt],
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[negative_prompt],
    )

    prompt_embeds = torch.cat(
        [prompt_embeds, child_tokens, father_tokens, mother_tokens],
        dim=1,
    )

    negative_prompt_embeds = torch.cat(
        [negative_prompt_embeds, zero_child_tokens, zero_father_tokens, zero_mother_tokens],
        dim=1,
    )

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    img = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        generator=generator,
    ).images[0]

    alpha_stats = collect_bap2_alpha_stats(pipe.unet)

    return img, alpha_stats

# ============================================================
# Evaluation loop
# ============================================================

rows = []

for triad_num, idx in enumerate(EVAL_IDXS, start=1):

    print(f"\n=== Triad {triad_num}/{len(EVAL_IDXS)} | idx={idx} ===")

    sample = val_ds[idx]

    batch = default_collate([sample])
    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    h_c = ensure_2d(batch["h_c"])
    h_p = ensure_2d(batch["h_p"])
    h_m = ensure_2d(batch["h_m"])

    h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
    h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
    h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    print("Father:", sample.get("key_p", "N/A"))
    print("Mother:", sample.get("key_m", "N/A"))
    print("Child :", sample.get("key_c", "N/A"))
    print("Gender:", gender)

    for parent_scale in PARENT_SCALES:
        for age in AGES:

            img, alpha_stats = generate_bap2_age(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age,
                parent_scale=parent_scale,
                seed=SEED,
            )

            h_gen = arcface_embed_from_pil(img)

            S_id = F.cosine_similarity(h_gen, h_c_eval).item()
            S_p = F.cosine_similarity(h_gen, h_p_eval).item()
            S_m = F.cosine_similarity(h_gen, h_m_eval).item()

            rows.append({
                "triad_idx": idx,
                "triad_num": triad_num,
                "variant": "BAP2_linear",
                "parent_scale": parent_scale,
                "age": age,
                "gender": gender,
                "S_id_child": S_id,
                "S_p_father": S_p,
                "S_m_mother": S_m,
                "S_family": 0.5 * (S_p + S_m),
                "balance_abs_gap": abs(S_p - S_m),
                "alpha_mean": alpha_stats["alpha_mean"] if alpha_stats else None,
                "alpha_std": alpha_stats["alpha_std"] if alpha_stats else None,
                "alpha_min": alpha_stats["alpha_min"] if alpha_stats else None,
                "alpha_max": alpha_stats["alpha_max"] if alpha_stats else None,
                "key_p": sample.get("key_p", ""),
                "key_m": sample.get("key_m", ""),
                "key_c": sample.get("key_c", ""),
            })

            print(
                f"λ={parent_scale} age={age} | "
                f"S_id={S_id:.4f} "
                f"S_p={S_p:.4f} "
                f"S_m={S_m:.4f} "
                f"S_family={0.5 * (S_p + S_m):.4f} "
                f"alpha_std={alpha_stats['alpha_std']:.5f}"
            )

df_bap2_eval_scales = pd.DataFrame(rows)

print("\n==============================")
print("Full results")
print("==============================")
display(df_bap2_eval_scales)

# ============================================================
# Global summary
# ============================================================

df_bap2_scale_summary = (
    df_bap2_eval_scales
    .groupby("parent_scale")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Global summary BAP2: λ=0.0 vs λ=0.1")
print("==============================")
display(df_bap2_scale_summary)

row_0 = df_bap2_scale_summary[df_bap2_scale_summary["parent_scale"] == 0.0].iloc[0]
row_1 = df_bap2_scale_summary[df_bap2_scale_summary["parent_scale"] == 0.1].iloc[0]

df_bap2_delta_scale_global = pd.DataFrame([{
    "delta_S_id_child_0.1_minus_0.0": row_1["S_id_child"] - row_0["S_id_child"],
    "delta_S_p_father_0.1_minus_0.0": row_1["S_p_father"] - row_0["S_p_father"],
    "delta_S_m_mother_0.1_minus_0.0": row_1["S_m_mother"] - row_0["S_m_mother"],
    "delta_S_family_0.1_minus_0.0": row_1["S_family"] - row_0["S_family"],
    "delta_balance_abs_gap_0.1_minus_0.0": row_1["balance_abs_gap"] - row_0["balance_abs_gap"],
}])

print("\n==============================")
print("Delta global BAP2: λ=0.1 menos λ=0.0")
print("==============================")
display(df_bap2_delta_scale_global)

# ============================================================
# Summary by age
# ============================================================

df_bap2_scale_age = (
    df_bap2_eval_scales
    .groupby(["age", "parent_scale"])
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Summary by age BAP2: λ=0.0 vs λ=0.1")
print("==============================")
display(df_bap2_scale_age)

delta_rows = []

for age in AGES:
    sub = df_bap2_scale_age[df_bap2_scale_age["age"] == age]

    r0 = sub[sub["parent_scale"] == 0.0].iloc[0]
    r1 = sub[sub["parent_scale"] == 0.1].iloc[0]

    delta_rows.append({
        "age": age,
        "delta_S_id_child_0.1_minus_0.0": r1["S_id_child"] - r0["S_id_child"],
        "delta_S_p_father_0.1_minus_0.0": r1["S_p_father"] - r0["S_p_father"],
        "delta_S_m_mother_0.1_minus_0.0": r1["S_m_mother"] - r0["S_m_mother"],
        "delta_S_family_0.1_minus_0.0": r1["S_family"] - r0["S_family"],
        "delta_balance_abs_gap_0.1_minus_0.0": r1["balance_abs_gap"] - r0["balance_abs_gap"],
    })

df_bap2_delta_scale_age = pd.DataFrame(delta_rows)

print("\n==============================")
print("Delta by age BAP2: λ=0.1 minus λ=0.0")
print("==============================")
display(df_bap2_delta_scale_age)

# ============================================================
# Save results
# ============================================================

EVAL_STEP = CURRENT_STEP if "CURRENT_STEP" in globals() else START_STEP

OUT_DIR = f"{DRIVE}/results/child_anchor_bap_gate/bap2_linear_step{EVAL_STEP}_lambda_ablation"
os.makedirs(OUT_DIR, exist_ok=True)

df_bap2_eval_scales.to_csv(f"{OUT_DIR}/df_bap2_eval_scales.csv", index=False)
df_bap2_scale_summary.to_csv(f"{OUT_DIR}/df_bap2_scale_summary.csv", index=False)
df_bap2_delta_scale_global.to_csv(f"{OUT_DIR}/df_bap2_delta_scale_global.csv", index=False)
df_bap2_scale_age.to_csv(f"{OUT_DIR}/df_bap2_scale_age.csv", index=False)
df_bap2_delta_scale_age.to_csv(f"{OUT_DIR}/df_bap2_delta_scale_age.csv", index=False)

print("\n✓ Results saved to:")
print(OUT_DIR)

EVAL_IDXS: [179, 12, 35, 48, 61, 74, 89, 103, 128, 150]

=== Triad 1/10 | idx=179 ===
Padre: F0854/MID1/P09020_face1.jpg
Madre: F0854/MID3/P09026_face0.jpg
Hijo : F0854/MID2/P09020_face0.jpg
Gender: man


  0%|          | 0/30 [00:00<?, ?it/s]

λ=0.0 age=15 | S_id=0.3175 S_p=0.0840 S_m=-0.0140 S_family=0.0350 alpha_std=0.00504


  0%|          | 0/30 [00:00<?, ?it/s]

λ=0.0 age=25 | S_id=0.3612 S_p=0.0805 S_m=-0.0065 S_family=0.0370 alpha_std=0.00514


...


Summary by age BAP2: λ=0.0 vs λ=0.1


,age,parent_scale,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap
0,15,0.0,0.143712,0.010130,0.036404,0.023267,0.089591
1,15,0.1,0.150570,0.018103,0.053007,0.035555,0.083967
2,25,0.0,0.176557,0.014231,0.029358,0.021794,0.066584
3,25,0.1,0.184387,0.021055,0.032742,0.026899,0.059249
4,40,0.0,0.261945,0.053046,0.028864,0.040955,0.060202
5,40,0.1,0.269313,0.057840,0.038925,0.048382,0.064634
6,60,0.0,0.172917,0.008662,0.022051,0.015356,0.062385
7,60,0.1,0.186896,0.007261,0.030732,0.018996,0.068210



Delta by age BAP2: λ=0.1 minus λ=0.0


,age,delta_S_id_child_0.1_minus_0.0,delta_S_p_father_0.1_minus_0.0,delta_S_m_mother_0.1_minus_0.0,delta_S_family_0.1_minus_0.0,delta_balance_abs_gap_0.1_minus_0.0
0,15,0.006858,0.007973,0.016603,0.012288,-0.005624
1,25,0.007829,0.006825,0.003384,0.005105,-0.007335
2,40,0.007368,0.004795,0.010061,0.007428,0.004431
3,60,0.013979,-0.001401,0.008681,0.003640,0.005825



✓ Results saved to:
/content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_linear_step500_lambda_ablation


In [12]:
# ============================================================
# Cell — BAP2 visual audit (FIXED)
# Does not use cross_attention_kwargs.
# Builds prompt_embeds + child/father/mother tokens as in evaluation.
# ============================================================

import os
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data._utils.collate import default_collate

# -------------------------------
# Configuration
# -------------------------------
TRIAD_IDX       = 0
AGES_TO_SHOW    = [5, 10, 15, 20, 25, 30, 40, 50, 60]
PARENT_SCALE    = 0.1
GUIDANCE_SCALE  = 4.0
NUM_STEPS       = 30
SEED            = 42

NEG_PROMPT = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"

print("CURRENT_STEP:", CURRENT_STEP if "CURRENT_STEP" in globals() else "N/A")
print("TRIAD_IDX:", TRIAD_IDX)
print("AGES:", AGES_TO_SHOW)
print("PARENT_SCALE:", PARENT_SCALE)

# -------------------------------
# Helpers
# -------------------------------
def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(unet, value):
    for proc in unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def load_fiw_image(key):
    path = os.path.join(FIW_ROOT, key)
    if not os.path.exists(path):
        print("Not found:", path)
        return None
    return Image.open(path).convert("RGB")

def tensor_img_to_pil(x):
    x = x.detach().cpu().float()
    x = (x + 1.0) / 2.0
    x = x.clamp(0, 1)
    x = x.permute(1, 2, 0).numpy()
    x = (x * 255).astype(np.uint8)
    return Image.fromarray(x)

def show_or_placeholder(ax, pil_img, title):
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    if pil_img is None:
        ax.text(0.5, 0.5, "Not found", ha="center", va="center")
    else:
        ax.imshow(pil_img)

@torch.inference_mode()
def generate_bap2_age_correct(h_c, h_p, h_m, gender, age, parent_scale, seed=42):
    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    set_parent_scale_bap2(pipe.unet, parent_scale)

    h_c = ensure_2d(h_c).to(device=DEVICE, dtype=DTYPE)
    h_p = ensure_2d(h_p).to(device=DEVICE, dtype=DTYPE)
    h_m = ensure_2d(h_m).to(device=DEVICE, dtype=DTYPE)

    child_tokens  = ip_model.image_proj_model(h_c)
    father_tokens = ip_model.image_proj_model(h_p)
    mother_tokens = ip_model.image_proj_model(h_m)

    zero_child_tokens  = ip_model.image_proj_model(torch.zeros_like(h_c))
    zero_father_tokens = ip_model.image_proj_model(torch.zeros_like(h_p))
    zero_mother_tokens = ip_model.image_proj_model(torch.zeros_like(h_m))

    prompt = f"portrait photo of a {gender}, age {age}, realistic, neutral expression"

    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=[prompt],
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[NEG_PROMPT],
    )

    prompt_embeds = torch.cat(
        [prompt_embeds, child_tokens, father_tokens, mother_tokens],
        dim=1,
    )

    negative_prompt_embeds = torch.cat(
        [negative_prompt_embeds, zero_child_tokens, zero_father_tokens, zero_mother_tokens],
        dim=1,
    )

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    img = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
    ).images[0]

    return img, prompt

# -------------------------------
# Load sample
# -------------------------------
sample = val_ds[TRIAD_IDX]

batch = default_collate([sample])
batch = {
    k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
    for k, v in batch.items()
}

h_c = ensure_2d(batch["h_c"])
h_p = ensure_2d(batch["h_p"])
h_m = ensure_2d(batch["h_m"])

gender = sample.get("gender", "person")
if gender not in ["man", "woman", "person"]:
    gender = "person"

img_p = load_fiw_image(sample["key_p"])
img_m = load_fiw_image(sample["key_m"])
img_c = load_fiw_image(sample["key_c"])

if img_c is None and "img" in sample:
    img_c = tensor_img_to_pil(sample["img"])

print("Father:", sample["key_p"])
print("Mother:", sample["key_m"])
print("Child :", sample["key_c"])
print("Gender:", gender)
print("Age real estimada:", sample["age"])

# -------------------------------
# Generate ages
# -------------------------------
gen_images = []

for age in AGES_TO_SHOW:
    img_gen, prompt = generate_bap2_age_correct(
        h_c=h_c,
        h_p=h_p,
        h_m=h_m,
        gender=gender,
        age=age,
        parent_scale=PARENT_SCALE,
        seed=SEED,
    )
    gen_images.append((age, img_gen))
    print(f"✓ age={age} generado")

# -------------------------------
# Visualization
# -------------------------------
total_panels = 3 + len(gen_images)
ncols = 4
nrows = math.ceil(total_panels / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.4 * nrows))
axes = np.array(axes).reshape(-1)

show_or_placeholder(axes[0], img_p, "Real father")
show_or_placeholder(axes[1], img_m, "Real mother")
show_or_placeholder(axes[2], img_c, f"Real child | age≈{sample['age']}")

for i, (age, img_gen) in enumerate(gen_images, start=3):
    show_or_placeholder(axes[i], img_gen, f"BAP2 age {age}")

for j in range(3 + len(gen_images), len(axes)):
    axes[j].axis("off")

plt.suptitle(
    f"BAP2 visual audit | idx={TRIAD_IDX} | gender={gender} | parent_scale={PARENT_SCALE}",
    fontsize=15
)
plt.tight_layout()
plt.show()

CURRENT_STEP: 500
TRIAD_IDX: 0
AGES: [5, 10, 15, 20, 25, 30, 40, 50, 60]
PARENT_SCALE: 0.1
Padre: F0011/MID1/P00114_face3.jpg
Madre: F0011/MID3/P00114_face1.jpg
Hijo : F0011/MID2/P00113_face1.jpg
Gender: man
Age real estimada: 5


  0%|          | 0/30 [00:00<?, ?it/s]

✓ age=5 generado


  0%|          | 0/30 [00:00<?, ?it/s]

✓ age=10 generado


...

  0%|          | 0/30 [00:00<?, ?it/s]

✓ age=50 generado


  0%|          | 0/30 [00:00<?, ?it/s]

✓ age=60 generado


<Figure size 1600x1320 with 12 Axes>

In [ ]:
# ============================================================
# Cell — parent_scale sweep BAP2 step500
# Clean triads: [1, 5, 8]
# Scales: [0.0, 0.05, 0.1, 0.2]
# Ages: [15, 25, 40, 60]
#
# No training. Only generates, evaluates and saves figures/tables.
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# -------------------------------
# Configuration
# -------------------------------
SWEEP_IDXS = [1, 5, 8]
PARENT_SCALES = [0.0, 0.05, 0.1, 0.2]
AGES = [15, 25, 40, 60]

GUIDANCE_SCALE = 4.0
NUM_STEPS = 30
SEED = 42

NEG_PROMPT = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"

EVAL_STEP = CURRENT_STEP if "CURRENT_STEP" in globals() else 500

OUT_SWEEP_DIR = (
    f"{DRIVE}/results/child_anchor_bap_gate/"
    f"bap2_step{EVAL_STEP}_parent_scale_sweep_clean"
)
os.makedirs(OUT_SWEEP_DIR, exist_ok=True)

print("EVAL_STEP:", EVAL_STEP)
print("SWEEP_IDXS:", SWEEP_IDXS)
print("PARENT_SCALES:", PARENT_SCALES)
print("AGES:", AGES)
print("Total generaciones:", len(SWEEP_IDXS) * len(PARENT_SCALES) * len(AGES))
print("Salida:", OUT_SWEEP_DIR)

# -------------------------------
# Helpers
# -------------------------------
def ensure_2d(x):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    return x

def set_parent_scale_bap2(unet, value):
    for proc in unet.attn_processors.values():
        if type(proc).__name__ == "BAP2":
            proc.parent_scale = value

def load_fiw_image_safe(key):
    path = os.path.join(FIW_ROOT, key)
    if not os.path.exists(path):
        return None
    return Image.open(path).convert("RGB")

def show_img(ax, img, title):
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    if img is None:
        ax.text(0.5, 0.5, "Not found", ha="center", va="center")
    else:
        ax.imshow(img)

def collect_bap2_alpha_stats(unet):
    vals = []

    for name, proc in unet.attn_processors.items():
        if type(proc).__name__ == "BAP2":
            if getattr(proc, "last_alpha", None) is not None:
                vals.append(proc.last_alpha.detach().float().cpu())

    if len(vals) == 0:
        return {
            "alpha_mean": None,
            "alpha_std": None,
            "alpha_min": None,
            "alpha_max": None,
        }

    x = torch.cat(vals, dim=0)

    return {
        "alpha_mean": x.mean().item(),
        "alpha_std": x.std().item(),
        "alpha_min": x.min().item(),
        "alpha_max": x.max().item(),
    }

# ArcFace preprocessing, same as in the previous evaluation
preprocess_arcface = T.Compose([
    T.Resize((112, 112)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

@torch.inference_mode()
def arcface_embed_from_pil(img):
    x = preprocess_arcface(img.convert("RGB")).unsqueeze(0).to(DEVICE)
    x = x.float()
    emb = arcface_torch(x)
    emb = F.normalize(emb, p=2, dim=1)
    return emb

@torch.inference_mode()
def generate_bap2_age_sweep(h_c, h_p, h_m, gender, age, parent_scale, seed=42):
    """
    Correct for our BAP2:
    encoder_hidden_states = [prompt | child_tokens | father_tokens | mother_tokens]
    """
    pipe.unet.eval()
    pipe.vae.eval()
    pipe.text_encoder.eval()
    ip_model.image_proj_model.eval()

    set_parent_scale_bap2(pipe.unet, parent_scale)

    h_c = ensure_2d(h_c).to(device=DEVICE, dtype=DTYPE)
    h_p = ensure_2d(h_p).to(device=DEVICE, dtype=DTYPE)
    h_m = ensure_2d(h_m).to(device=DEVICE, dtype=DTYPE)

    child_tokens  = ip_model.image_proj_model(h_c)
    father_tokens = ip_model.image_proj_model(h_p)
    mother_tokens = ip_model.image_proj_model(h_m)

    zero_child_tokens  = ip_model.image_proj_model(torch.zeros_like(h_c))
    zero_father_tokens = ip_model.image_proj_model(torch.zeros_like(h_p))
    zero_mother_tokens = ip_model.image_proj_model(torch.zeros_like(h_m))

    prompt = f"portrait photo of a {gender}, age {age}, realistic, neutral expression"

    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=[prompt],
        device=DEVICE,
        num_images_per_prompt=1,
        do_classifier_free_guidance=True,
        negative_prompt=[NEG_PROMPT],
    )

    prompt_embeds = torch.cat(
        [prompt_embeds, child_tokens, father_tokens, mother_tokens],
        dim=1,
    )

    negative_prompt_embeds = torch.cat(
        [negative_prompt_embeds, zero_child_tokens, zero_father_tokens, zero_mother_tokens],
        dim=1,
    )

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    img = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
    ).images[0]

    alpha_stats = collect_bap2_alpha_stats(pipe.unet)

    return img, prompt, alpha_stats

# ============================================================
# Sweep
# ============================================================

all_rows = []

for triad_idx in SWEEP_IDXS:

    print("\n" + "=" * 100)
    print("TRIAD_IDX:", triad_idx)
    print("=" * 100)

    sample = val_ds[triad_idx]

    batch = default_collate([sample])
    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    h_c = ensure_2d(batch["h_c"])
    h_p = ensure_2d(batch["h_p"])
    h_m = ensure_2d(batch["h_m"])

    h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
    h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
    h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    img_p = load_fiw_image_safe(sample["key_p"])
    img_m = load_fiw_image_safe(sample["key_m"])
    img_c = load_fiw_image_safe(sample["key_c"])

    print("Father:", sample["key_p"])
    print("Mother:", sample["key_m"])
    print("Child :", sample["key_c"])
    print("Gender:", gender)
    print("Age real estimada:", sample["age"])

    generated = {}

    for parent_scale in PARENT_SCALES:
        for age in AGES:

            img_gen, prompt, alpha_stats = generate_bap2_age_sweep(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age,
                parent_scale=parent_scale,
                seed=SEED,
            )

            generated[(parent_scale, age)] = img_gen

            h_gen = arcface_embed_from_pil(img_gen)

            S_id = F.cosine_similarity(h_gen, h_c_eval).item()
            S_p = F.cosine_similarity(h_gen, h_p_eval).item()
            S_m = F.cosine_similarity(h_gen, h_m_eval).item()

            all_rows.append({
                "triad_idx": triad_idx,
                "gender": gender,
                "age": age,
                "parent_scale": parent_scale,
                "S_id_child": S_id,
                "S_p_father": S_p,
                "S_m_mother": S_m,
                "S_family": 0.5 * (S_p + S_m),
                "balance_abs_gap": abs(S_p - S_m),
                "alpha_mean": alpha_stats["alpha_mean"],
                "alpha_std": alpha_stats["alpha_std"],
                "alpha_min": alpha_stats["alpha_min"],
                "alpha_max": alpha_stats["alpha_max"],
                "key_p": sample["key_p"],
                "key_m": sample["key_m"],
                "key_c": sample["key_c"],
            })

            print(
                f"idx={triad_idx} | λ={parent_scale} | age={age} | "
                f"S_id={S_id:.4f} | S_family={0.5 * (S_p + S_m):.4f}"
            )

    # -------------------------------
    # Figure per triad
    # Fila 0: reales
    # Following rows: each parent_scale
    # Columns: real/ages
    # -------------------------------
    nrows = 1 + len(PARENT_SCALES)
    ncols = max(3, len(AGES))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.2 * nrows))

    if nrows == 1:
        axes = axes.reshape(1, -1)

    # Fila reales
    show_img(axes[0, 0], img_p, "Real father")
    show_img(axes[0, 1], img_m, "Real mother")
    show_img(axes[0, 2], img_c, f"Real child | age≈{sample['age']}")

    for c in range(3, ncols):
        axes[0, c].axis("off")

    # Scale rows
    for r, parent_scale in enumerate(PARENT_SCALES, start=1):
        for c, age in enumerate(AGES):
            img_gen = generated[(parent_scale, age)]
            show_img(
                axes[r, c],
                img_gen,
                f"λ={parent_scale} | age {age}"
            )

    plt.suptitle(
        f"Sweep parent_scale BAP2 | idx={triad_idx} | gender={gender} | step={EVAL_STEP}",
        fontsize=15
    )
    plt.tight_layout()

    fig_path = f"{OUT_SWEEP_DIR}/sweep_idx{triad_idx}_step{EVAL_STEP}.png"
    plt.savefig(fig_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("✓ Figura guardada:", fig_path)

# ============================================================
# Tablas resumen
# ============================================================

df_sweep = pd.DataFrame(all_rows)

csv_path = f"{OUT_SWEEP_DIR}/df_parent_scale_sweep_clean.csv"
df_sweep.to_csv(csv_path, index=False)

print("\n✓ Full CSV saved:")
print(csv_path)

print("\n==============================")
print("Full results")
print("==============================")
display(df_sweep)

summary_global = (
    df_sweep
    .groupby("parent_scale")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Global summary by parent_scale")
print("==============================")
display(summary_global)

summary_age = (
    df_sweep
    .groupby(["age", "parent_scale"])
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Summary by age and parent_scale")
print("==============================")
display(summary_age)

# Delta contra λ=0.0
base = summary_global[summary_global["parent_scale"] == 0.0].iloc[0]

delta_rows = []
for _, row in summary_global.iterrows():
    delta_rows.append({
        "parent_scale": row["parent_scale"],
        "delta_S_id_child_vs_0.0": row["S_id_child"] - base["S_id_child"],
        "delta_S_family_vs_0.0": row["S_family"] - base["S_family"],
        "delta_S_p_father_vs_0.0": row["S_p_father"] - base["S_p_father"],
        "delta_S_m_mother_vs_0.0": row["S_m_mother"] - base["S_m_mother"],
        "delta_balance_abs_gap_vs_0.0": row["balance_abs_gap"] - base["balance_abs_gap"],
    })

df_delta = pd.DataFrame(delta_rows)

print("\n==============================")
print("Delta global contra parent_scale=0.0")
print("==============================")
display(df_delta)

summary_global.to_csv(f"{OUT_SWEEP_DIR}/summary_global_parent_scale.csv", index=False)
summary_age.to_csv(f"{OUT_SWEEP_DIR}/summary_age_parent_scale.csv", index=False)
df_delta.to_csv(f"{OUT_SWEEP_DIR}/delta_vs_scale0.csv", index=False)

print("\n✓ Summaries saved to:")
print(OUT_SWEEP_DIR)

EVAL_STEP: 500
SWEEP_IDXS: [1, 5, 8]
PARENT_SCALES: [0.0, 0.05, 0.1, 0.2]
AGES: [15, 25, 40, 60]
Total generaciones: 48
Salida: /content/drive/MyDrive/Notebook4/results/child_anchor_bap_gate/bap2_step500_parent_scale_sweep_clean

TRIAD_IDX: 1
Padre: F0037/MID1/P00382_face3.jpg
Madre: F0037/MID5/P00384_face1.jpg
Hijo : F0037/MID6/P00385_face6.jpg
Gender: man
Age real estimada: 8


  0%|          | 0/30 [00:00<?, ?it/s]

idx=1 | λ=0.0 | age=15 | S_id=0.2669 | S_family=0.0139


  0%|          | 0/30 [00:00<?, ?it/s]

idx=1 | λ=0.0 | age=25 | S_id=0.3444 | S_family=-0.0096


...

idx=5 | λ=0.2 | age=25 | S_id=0.0911 | S_family=-0.0079


  0%|          | 0/30 [00:00<?, ?it/s]

idx=5 | λ=0.2 | age=40 | S_id=0.1343 | S_family=0.0210


  0%|          | 0/30 [00:00<?, ?it/s]

idx=5 | λ=0.2 | age=60 | S_id=0.1759 | S_family=0.0625


In [8]:
from collections import Counter
import os
import pandas as pd

print("test_ds exists:", "test_ds" in globals())
print("bap2_modules exists:", "bap2_modules" in globals())
print("generate_bap2_final exists:", "generate_bap2_final" in globals())
print("arcface_embed_from_pil exists:", "arcface_embed_from_pil" in globals())
print("ensure_2d exists:", "ensure_2d" in globals())

if "bap2_modules" in globals():
    print("len bap2_modules:", len(bap2_modules))

types = Counter(type(p).__name__ for p in pipe.unet.attn_processors.values())
print("Processors:", types)

OUT_CSV = f"{DRIVE}/results/eval/nogate/nogate.csv"
print("Partial CSV exists:", os.path.exists(OUT_CSV))

if os.path.exists(OUT_CSV):
    df = pd.read_csv(OUT_CSV)
    print("Filas actuales:", len(df))
    print("Unique triads:", df["triad_idx"].nunique())
    print("Last rows:")
    display(df.tail())

test_ds exists: True
bap2_modules exists: True
generate_bap2_final exists: False
arcface_embed_from_pil exists: False
ensure_2d exists: False
len bap2_modules: 16
Processors: Counter({'LoRAAttnProcessor2_0': 16, 'BAP2': 16})
Partial CSV exists: True
Filas actuales: 450
Unique triads: 90
Last rows:


,variant,checkpoint_step,triad_idx,age,parent_scale,gender,S_id_child,S_p_father,S_m_mother,S_family,...,alpha_mean,alpha_std,alpha_min,alpha_max,key_p,key_m,key_c,seed,prompt,image_path
445,NoGate_fixed_alpha05,fixed_alpha_0.5,89,5,0.1,woman,0.328934,0.086942,-0.005537,0.040703,...,0.5,0.0,0.5,0.5,F0542/MID4/P05684_face5.jpg,F0542/MID5/P05684_face2.jpg,F0542/MID2/P05684_face1.jpg,89047,"portrait photo of a woman, age 5, realistic, n...",/content/drive/MyDrive/Notebook4/results/final...
446,NoGate_fixed_alpha05,fixed_alpha_0.5,89,15,0.1,woman,0.357122,0.074940,-0.046193,0.014374,...,0.5,0.0,0.5,0.5,F0542/MID4/P05684_face5.jpg,F0542/MID5/P05684_face2.jpg,F0542/MID2/P05684_face1.jpg,89057,"portrait photo of a woman, age 15, realistic, ...",/content/drive/MyDrive/Notebook4/results/final...
447,NoGate_fixed_alpha05,fixed_alpha_0.5,89,25,0.1,woman,0.432372,0.133125,-0.077023,0.028051,...,0.5,0.0,0.5,0.5,F0542/MID4/P05684_face5.jpg,F0542/MID5/P05684_face2.jpg,F0542/MID2/P05684_face1.jpg,89067,"portrait photo of a woman, age 25, realistic, ...",/content/drive/MyDrive/Notebook4/results/final...
448,NoGate_fixed_alpha05,fixed_alpha_0.5,89,40,0.1,woman,0.351275,0.065148,0.012005,0.038576,...,0.5,0.0,0.5,0.5,F0542/MID4/P05684_face5.jpg,F0542/MID5/P05684_face2.jpg,F0542/MID2/P05684_face1.jpg,89082,"portrait photo of a woman, age 40, realistic, ...",/content/drive/MyDrive/Notebook4/results/final...
449,NoGate_fixed_alpha05,fixed_alpha_0.5,89,60,0.1,woman,-0.061819,-0.047405,0.026627,-0.010389,...,0.5,0.0,0.5,0.5,F0542/MID4/P05684_face5.jpg,F0542/MID5/P05684_face2.jpg,F0542/MID2/P05684_face1.jpg,89102,"portrait photo of a woman, age 60, realistic, ...",/content/drive/MyDrive/Notebook4/results/final...


In [10]:
# ============================================================
# TEST FINAL — NoGate / FixedGate α=0.5
# 208 triads × 5 ages × λ=0.1 = 1040 images
#
# Baseline:
#   child anchor + father + mother
#   but WITHOUT a learned gate
#   alpha = 0.5 fijo
# ============================================================

import os
import torch
import torch.nn.functional as F
import pandas as pd
from torch.utils.data._utils.collate import default_collate
from IPython.display import display

# ------------------------------------------------------------
# Verificaciones
# ------------------------------------------------------------

required = [
    "test_ds",
    "bap2_modules",
    "generate_bap2_final",
    "arcface_embed_from_pil",
    "ensure_2d",
]

missing = [x for x in required if x not in globals()]
assert len(missing) == 0, f"Missing objects in memory: {missing}"

assert len(bap2_modules) == 16, "bap2_modules must have 16 modules."

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

AGES = [5, 15, 25, 40, 60]
PARENT_SCALE = 0.1

BASE_SEED = 42

OUT_DIR = f"{DRIVE}/results/eval/nogate"
IMG_DIR = f"{OUT_DIR}/images"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(IMG_DIR, exist_ok=True)

OUT_CSV = f"{OUT_DIR}/nogate.csv"

print("Test triads:", len(test_ds))
print("Ages:", AGES)
print("Parent scale:", PARENT_SCALE)
print("Total images:", len(test_ds) * len(AGES))
print("OUT_DIR:", OUT_DIR)

# ------------------------------------------------------------
# Forzar NoGate: alpha = sigmoid(0) = 0.5
# ------------------------------------------------------------

for name, proc in bap2_modules.items():
    proc.gate.float()
    with torch.no_grad():
        proc.gate.weight.zero_()
        proc.gate.bias.zero_()
    proc.gate.to(DEVICE)

print("✓ Gates reseteados a cero: alpha fijo = 0.5")

# Sanity check alpha
with torch.no_grad():
    first_name, first_proc = next(iter(bap2_modules.items()))
    dummy = torch.randn(
        1,
        first_proc.gate.in_features,
        device=DEVICE,
        dtype=torch.float32,
    )
    alpha = torch.sigmoid(first_proc.gate(dummy))

print("Alpha sanity:")
print("mean:", alpha.mean().item())
print("std :", alpha.std().item())
print("min :", alpha.min().item())
print("max :", alpha.max().item())

assert abs(alpha.mean().item() - 0.5) < 1e-6
assert alpha.std().item() < 1e-6

# ------------------------------------------------------------
# Saving NoGate images
# ------------------------------------------------------------

def save_nogate_image(img, triad_idx, age):
    subdir = f"{IMG_DIR}/age_{age}"
    os.makedirs(subdir, exist_ok=True)

    path = f"{subdir}/t{triad_idx:04d}_a{int(age):02d}.jpg"
    img.save(path, quality=95)
    return path

# ------------------------------------------------------------
# Automatic resume
# ------------------------------------------------------------

rows = []
done = set()

if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV)
    rows = old.to_dict("records")
    done = set(zip(old["triad_idx"], old["age"]))
    print(f"✓ Resume detected: {len(old)} existing rows")

# ------------------------------------------------------------
# Loop principal
# ------------------------------------------------------------

for triad_idx in range(len(test_ds)):

    sample = test_ds[triad_idx]

    batch = default_collate([sample])
    batch = {
        k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }

    h_c = ensure_2d(batch["h_c"])
    h_p = ensure_2d(batch["h_p"])
    h_m = ensure_2d(batch["h_m"])

    h_c_eval = F.normalize(h_c.float(), p=2, dim=1)
    h_p_eval = F.normalize(h_p.float(), p=2, dim=1)
    h_m_eval = F.normalize(h_m.float(), p=2, dim=1)

    gender = sample.get("gender", "person")
    if gender not in ["man", "woman", "person"]:
        gender = "person"

    print(f"\nNoGate | Triada {triad_idx + 1}/{len(test_ds)} | gender={gender}")

    for age in AGES:

        if (triad_idx, age) in done:
            print(f"skip triad={triad_idx} age={age}")
            continue

        paired_seed = BASE_SEED + triad_idx * 1000 + age

        img, alpha_stats, prompt = generate_bap2_final(
            h_c=h_c,
            h_p=h_p,
            h_m=h_m,
            gender=gender,
            age=age,
            parent_scale=PARENT_SCALE,
            seed=paired_seed,
        )

        img_path = save_nogate_image(
            img=img,
            triad_idx=triad_idx,
            age=age,
        )

        h_gen = arcface_embed_from_pil(img)

        S_id = F.cosine_similarity(h_gen, h_c_eval).item()
        S_p  = F.cosine_similarity(h_gen, h_p_eval).item()
        S_m  = F.cosine_similarity(h_gen, h_m_eval).item()

        row = {
            "variant": "NoGate_fixed_alpha05",
            "checkpoint_step": "fixed_alpha_0.5",
            "triad_idx": triad_idx,
            "age": age,
            "parent_scale": PARENT_SCALE,
            "gender": gender,
            "S_id_child": S_id,
            "S_p_father": S_p,
            "S_m_mother": S_m,
            "S_family": 0.5 * (S_p + S_m),
            "balance_abs_gap": abs(S_p - S_m),
            "alpha_mean": alpha_stats["alpha_mean"],
            "alpha_std": alpha_stats["alpha_std"],
            "alpha_min": alpha_stats["alpha_min"],
            "alpha_max": alpha_stats["alpha_max"],
            "key_p": sample.get("key_p", ""),
            "key_m": sample.get("key_m", ""),
            "key_c": sample.get("key_c", ""),
            "seed": paired_seed,
            "prompt": prompt,
            "image_path": img_path,
        }

        rows.append(row)

        # Incremental saving
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

        print(
            f"age={age} | "
            f"S_id={S_id:.4f} | "
            f"S_p={S_p:.4f} | "
            f"S_m={S_m:.4f} | "
            f"S_family={row['S_family']:.4f} | "
            f"gap={row['balance_abs_gap']:.4f} | "
            f"alpha_mean={alpha_stats['alpha_mean']:.4f}"
        )

# ------------------------------------------------------------
# Final summary NoGate
# ------------------------------------------------------------

df_final_nogate = pd.DataFrame(rows)

print("\n==============================")
print("Filas finales NoGate")
print("==============================")
print(len(df_final_nogate))

expected_rows = len(test_ds) * len(AGES)
print("Esperadas:", expected_rows)

if len(df_final_nogate) != expected_rows:
    print("⚠ Warning: unexpected number of rows. Check whether there was an interruption.")
else:
    print("✓ Row count is correct.")

summary_nogate_global = (
    df_final_nogate
    .groupby("variant")
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

summary_nogate_age = (
    df_final_nogate
    .groupby(["age"])
    [[
        "S_id_child",
        "S_p_father",
        "S_m_mother",
        "S_family",
        "balance_abs_gap",
        "alpha_mean",
        "alpha_std",
        "alpha_min",
        "alpha_max",
    ]]
    .mean()
    .reset_index()
)

print("\n==============================")
print("Global summary NoGate")
print("==============================")
display(summary_nogate_global)

print("\n==============================")
print("Summary by age NoGate")
print("==============================")
display(summary_nogate_age)

summary_nogate_global.to_csv(f"{OUT_DIR}/summary_global_nogate.csv", index=False)
summary_nogate_age.to_csv(f"{OUT_DIR}/summary_age_nogate.csv", index=False)

print("\n✓ TEST FINAL NoGate terminado")
print("CSV:", OUT_CSV)
print("Images:", IMG_DIR)
print("Summary:", OUT_DIR)

Test triads: 208
Edades: [5, 15, 25, 40, 60]
Parent scale: 0.1
Total images: 1040
OUT_DIR: /content/drive/MyDrive/Notebook4/results/eval/nogate
✓ Gates reseteados a cero: alpha fijo = 0.5
Alpha sanity:
mean: 0.5
std : 0.0
min : 0.5
max : 0.5
✓ Resume detected: 450 existing rows

NoGate | Triada 1/208 | gender=woman
skip triad=0 age=5
skip triad=0 age=15
skip triad=0 age=25
skip triad=0 age=40
skip triad=0 age=60

NoGate | Triada 2/208 | gender=woman
skip triad=1 age=5
skip triad=1 age=15
skip triad=1 age=25
skip triad=1 age=40
skip triad=1 age=60

NoGate | Triada 3/208 | gender=woman
skip triad=2 age=5
skip triad=2 age=15
skip triad=2 age=25
skip triad=2 age=40
skip triad=2 age=60

NoGate | Triada 4/208 | gender=woman
skip triad=3 age=5
skip triad=3 age=15
skip triad=3 age=25
skip triad=3 age=40
skip triad=3 age=60

NoGate | Triada 5/208 | gender=man
skip triad=4 age=5
skip triad=4 age=15
skip triad=4 age=25
skip triad=4 age=40
skip triad=4 age=60

NoGate | Triada 6/208 | gender=man
sk

  0%|          | 0/30 [00:00<?, ?it/s]

age=5 | S_id=0.4076 | S_p=0.0797 | S_m=0.0045 | S_family=0.0421 | gap=0.0752 | alpha_mean=0.5000


  0%|          | 0/30 [00:00<?, ?it/s]

age=15 | S_id=0.4859 | S_p=0.0347 | S_m=0.0221 | S_family=0.0284 | gap=0.0126 | alpha_mean=0.5000


...

age=60 | S_id=0.2367 | S_p=0.0057 | S_m=0.0589 | S_family=0.0323 | gap=0.0532 | alpha_mean=0.5000

Filas finales NoGate
1040
Esperadas: 1040
✓ Row count is correct.

Global summary NoGate


,variant,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap,alpha_mean,alpha_std,alpha_min,alpha_max
0,NoGate_fixed_alpha05,0.186611,0.016898,0.022914,0.019906,0.059253,0.5,0.0,0.5,0.5



Summary by age NoGate


,age,S_id_child,S_p_father,S_m_mother,S_family,balance_abs_gap,alpha_mean,alpha_std,alpha_min,alpha_max
0,5,0.228732,0.021340,0.021394,0.021367,0.058969,0.5,0.0,0.5,0.5
1,15,0.174104,0.014445,0.021142,0.017793,0.051344,0.5,0.0,0.5,0.5
2,25,0.174422,0.011256,0.016773,0.014014,0.056325,0.5,0.0,0.5,0.5
3,40,0.175843,0.015316,0.024331,0.019823,0.061478,0.5,0.0,0.5,0.5
4,60,0.179955,0.022136,0.030933,0.026535,0.068150,0.5,0.0,0.5,0.5



✓ TEST FINAL NoGate terminado
CSV: /content/drive/MyDrive/Notebook4/results/eval/nogate/nogate.csv
Images: /content/drive/MyDrive/Notebook4/results/eval/nogate/img
Summary: /content/drive/MyDrive/Notebook4/results/eval/nogate


In [2]:
# ============================================================
# Final comparison:
# BAP2 step4689 λ=0.1 vs NoGate fixed α=0.5 λ=0.1
# CPU only
# ============================================================

import os
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from scipy.stats import wilcoxon
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    print("⚠ scipy not available:", e)

BAP2_CSV = f"{DRIVE}/results/eval/bap2/bap2.csv"
NOGATE_CSV = f"{DRIVE}/results/eval/nogate/nogate.csv"

assert os.path.exists(BAP2_CSV), BAP2_CSV
assert os.path.exists(NOGATE_CSV), NOGATE_CSV

df_bap2 = pd.read_csv(BAP2_CSV)
df_nogate = pd.read_csv(NOGATE_CSV)

# Only BAP2 with active parents
df_bap2_01 = df_bap2[df_bap2["parent_scale"] == 0.1].copy()

print("BAP2 λ=0.1 rows:", len(df_bap2_01))
print("NoGate rows:", len(df_nogate))

metrics = [
    "S_id_child",
    "S_p_father",
    "S_m_mother",
    "S_family",
    "balance_abs_gap",
]

key_cols = ["triad_idx", "age"]

merged = df_bap2_01[key_cols + metrics].merge(
    df_nogate[key_cols + metrics],
    on=key_cols,
    suffixes=("_bap2", "_nogate")
)

print("Pares comparados:", len(merged))
print("Triads:", merged["triad_idx"].nunique())
print("Ages:", sorted(merged["age"].unique()))

assert len(merged) == 1040, "Expected 208 triads × 5 ages = 1040 pairs"

rng = np.random.default_rng(123)
B = 5000

rows = []

for m in metrics:
    d = merged[f"{m}_bap2"] - merged[f"{m}_nogate"]
    arr = d.values

    boot = []
    for _ in range(B):
        sample = rng.choice(arr, size=len(arr), replace=True)
        boot.append(sample.mean())

    boot = np.array(boot)

    row = {
        "metric": m,
        "mean_bap2": merged[f"{m}_bap2"].mean(),
        "mean_nogate": merged[f"{m}_nogate"].mean(),
        "mean_delta_bap2_minus_nogate": d.mean(),
        "median_delta": d.median(),
        "ci95_low": np.percentile(boot, 2.5),
        "ci95_high": np.percentile(boot, 97.5),
        "win_rate_bap2_gt_nogate": (d > 0).mean(),
        "n_pairs": len(d),
    }

    if HAS_SCIPY:
        try:
            stat, p = wilcoxon(d)
            row["wilcoxon_stat"] = stat
            row["wilcoxon_p"] = p
        except Exception as e:
            row["wilcoxon_stat"] = None
            row["wilcoxon_p"] = None
            row["wilcoxon_error"] = str(e)

    rows.append(row)

df_gate_stats = pd.DataFrame(rows)

print("\n==============================")
print("BAP2 λ=0.1 vs NoGate α=0.5")
print("==============================")
display(df_gate_stats)

# By age
age_rows = []

for age in sorted(merged["age"].unique()):
    sub = merged[merged["age"] == age]

    for m in metrics:
        d = sub[f"{m}_bap2"] - sub[f"{m}_nogate"]

        age_rows.append({
            "age": age,
            "metric": m,
            "mean_delta_bap2_minus_nogate": d.mean(),
            "median_delta": d.median(),
            "win_rate_bap2_gt_nogate": (d > 0).mean(),
            "n_pairs": len(d),
        })

df_gate_stats_age = pd.DataFrame(age_rows)

print("\n==============================")
print("BAP2 vs NoGate by age")
print("==============================")
display(df_gate_stats_age)

OUT_DIR = f"{DRIVE}/results/eval/cmp"
os.makedirs(OUT_DIR, exist_ok=True)

df_gate_stats.to_csv(f"{OUT_DIR}/stats_bap2_vs_nogate.csv", index=False)
df_gate_stats_age.to_csv(f"{OUT_DIR}/stats_bap2_vs_nogate_by_age.csv", index=False)

print("\n✓ Saved to:", OUT_DIR)

BAP2 λ=0.1 rows: 1040
NoGate rows: 1040
Pares comparados: 1040
Triads: 208
Edades: [np.int64(5), np.int64(15), np.int64(25), np.int64(40), np.int64(60)]

BAP2 λ=0.1 vs NoGate α=0.5


,metric,mean_bap2,mean_nogate,mean_delta_bap2_minus_nogate,median_delta,ci95_low,ci95_high,win_rate_bap2_gt_nogate,n_pairs,wilcoxon_stat,wilcoxon_p
0,S_id_child,0.186964,0.186611,0.000353,0.000132,-0.000110,0.000870,0.531731,1040,253717.0,0.080341
1,S_p_father,0.017099,0.016898,0.000200,0.000092,-0.000114,0.000507,0.513462,1040,253317.0,0.073455
2,S_m_mother,0.023106,0.022914,0.000192,0.000082,-0.000107,0.000492,0.524038,1040,252571.0,0.061903
3,S_family,0.020102,0.019906,0.000196,0.000087,-0.000056,0.000423,0.529808,1040,244683.0,0.007338
4,balance_abs_gap,0.058842,0.059253,-0.000411,-0.000073,-0.000781,-0.000055,0.484615,1040,255640.0,0.121085



BAP2 vs NoGate by age


,age,metric,mean_delta_bap2_minus_nogate,median_delta,win_rate_bap2_gt_nogate,n_pairs
0,5,S_id_child,0.000470,0.000232,0.581731,208
1,5,S_p_father,0.000091,0.000123,0.528846,208
2,5,S_m_mother,0.000379,0.000193,0.552885,208
3,5,S_family,0.000235,0.000125,0.548077,208
4,5,balance_abs_gap,-0.001093,-0.000048,0.490385,208
5,15,S_id_child,0.000341,0.000112,0.528846,208
6,15,S_p_father,0.000028,-0.000022,0.495192,208
7,15,S_m_mother,0.000582,0.000215,0.567308,208
8,15,S_family,0.000305,0.000065,0.514423,208
9,15,balance_abs_gap,0.000253,0.000111,0.514423,208



✓ Saved to: /content/drive/MyDrive/Notebook4/results/eval/cmp


In [ ]:
# ============================================================
# FIGURE A — Qualitative comparison by method
#
# Columnas:
# Father | Mother | Real Child | BAP2 λ=0.0 | NoGate α=0.5 | BAP2 λ=0.1
#
# Fixed age: 25
# Selection: 8 representative triads
# Output: PNG + PDF + CSV of selected triads
# ============================================================

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw

# ------------------------------------------------------------
# Rutas
# ------------------------------------------------------------

BAP2_CSV = f"{DRIVE}/results/eval/bap2/bap2.csv"
NOGATE_CSV = f"{DRIVE}/results/eval/nogate/nogate.csv"

OUT_DIR = f"{DRIVE}/results/figs"
os.makedirs(OUT_DIR, exist_ok=True)

FIG_PNG = f"{OUT_DIR}/method/a25.png"
SEL_CSV = f"{OUT_DIR}/meta/method_sel.csv"

assert os.path.exists(BAP2_CSV), BAP2_CSV
assert os.path.exists(NOGATE_CSV), NOGATE_CSV
assert "FIW_ROOT" in globals(), "Falta FIW_ROOT"

df_bap2 = pd.read_csv(BAP2_CSV)
df_nogate = pd.read_csv(NOGATE_CSV)

AGE_FIG = 25
N_ROWS = 8

# ------------------------------------------------------------
# Preparar datos
# ------------------------------------------------------------

b0 = df_bap2[(df_bap2["age"] == AGE_FIG) & (df_bap2["parent_scale"] == 0.0)].copy()
b1 = df_bap2[(df_bap2["age"] == AGE_FIG) & (df_bap2["parent_scale"] == 0.1)].copy()
ng = df_nogate[df_nogate["age"] == AGE_FIG].copy()

metrics = [
    "S_id_child",
    "S_p_father",
    "S_m_mother",
    "S_family",
    "balance_abs_gap",
    "image_path",
    "key_p",
    "key_m",
    "key_c",
    "gender",
]

merged = (
    b0[["triad_idx"] + metrics]
    .merge(
        b1[["triad_idx"] + metrics],
        on="triad_idx",
        suffixes=("_lambda0", "_bap2")
    )
    .merge(
        ng[["triad_idx"] + metrics],
        on="triad_idx",
        suffixes=("", "_nogate")
    )
)

# Rename NoGate columns left without suffix after the second merge
rename_ng = {
    "S_id_child": "S_id_child_nogate",
    "S_p_father": "S_p_father_nogate",
    "S_m_mother": "S_m_mother_nogate",
    "S_family": "S_family_nogate",
    "balance_abs_gap": "balance_abs_gap_nogate",
    "image_path": "image_path_nogate",
    "key_p": "key_p_nogate",
    "key_m": "key_m_nogate",
    "key_c": "key_c_nogate",
    "gender": "gender_nogate",
}
merged = merged.rename(columns=rename_ng)

# Delta parental BAP2 vs lambda0
merged["delta_family_bap2_vs_lambda0"] = (
    merged["S_family_bap2"] - merged["S_family_lambda0"]
)

merged["delta_id_bap2_vs_lambda0"] = (
    merged["S_id_child_bap2"] - merged["S_id_child_lambda0"]
)

merged["delta_family_bap2_vs_nogate"] = (
    merged["S_family_bap2"] - merged["S_family_nogate"]
)

# ------------------------------------------------------------
# Representative selection
# ------------------------------------------------------------
# Condiciones:
# 1) BAP2 mejora S_family frente a λ=0.0
# 2) S_id_child_bap2 is not too low
# 3) avoid overly extreme examples: use a combined ranking

pool = merged[
    (merged["delta_family_bap2_vs_lambda0"] > 0)
    & (merged["S_id_child_bap2"] > merged["S_id_child_bap2"].quantile(0.40))
].copy()

if len(pool) < N_ROWS:
    print("⚠ Small pool, relaxing the S_id criterion.")
    pool = merged[merged["delta_family_bap2_vs_lambda0"] > 0].copy()

if len(pool) < N_ROWS:
    print("⚠ Pool still small, using all.")
    pool = merged.copy()

# score moderado:
# high S_id + high family gain, without taking only extremes
pool["rank_id"] = pool["S_id_child_bap2"].rank(pct=True)
pool["rank_delta_family"] = pool["delta_family_bap2_vs_lambda0"].rank(pct=True)
pool["score"] = 0.55 * pool["rank_id"] + 0.45 * pool["rank_delta_family"]

# Select from high but spread percentiles, not just the top 8
pool_sorted = pool.sort_values("score", ascending=True).reset_index(drop=True)

target_qs = np.linspace(0.60, 0.95, N_ROWS)
selected_idxs = []
for q in target_qs:
    pos = int(q * (len(pool_sorted) - 1))
    selected_idxs.append(pos)

selected = pool_sorted.iloc[selected_idxs].drop_duplicates("triad_idx")

# If rows are missing due to duplicates, fill with the best remaining
if len(selected) < N_ROWS:
    extra = pool_sorted[~pool_sorted["triad_idx"].isin(selected["triad_idx"])]
    selected = pd.concat([selected, extra.tail(N_ROWS - len(selected))], axis=0)

selected = selected.head(N_ROWS).copy()
selected = selected.sort_values("triad_idx").reset_index(drop=True)

print("Selected triads:")
display(selected[[
    "triad_idx",
    "S_id_child_lambda0",
    "S_id_child_nogate",
    "S_id_child_bap2",
    "S_family_lambda0",
    "S_family_nogate",
    "S_family_bap2",
    "delta_family_bap2_vs_lambda0",
    "delta_family_bap2_vs_nogate",
]])

selected.to_csv(SEL_CSV, index=False)

# ------------------------------------------------------------
# Image helpers
# ------------------------------------------------------------

def read_img(path):
    try:
        img = Image.open(path).convert("RGB")
        return img
    except Exception as e:
        print("⚠ Could not open:", path, "|", e)
        return Image.new("RGB", (256, 256), color=(230, 230, 230))

def square_center_crop(img):
    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    img = img.crop((left, top, left + side, top + side))
    return img

def prep_img(path, size=180):
    img = read_img(path)
    img = square_center_crop(img)
    img = ImageOps.fit(img, (size, size), method=Image.Resampling.LANCZOS)
    return img

def add_border(img, width=2):
    return ImageOps.expand(img, border=width, fill=(255, 255, 255))

def key_to_path(key):
    return os.path.join(FIW_ROOT, key)

# ------------------------------------------------------------
# Crear figura
# ------------------------------------------------------------

columns = [
    "Father",
    "Mother",
    "Real Child",
    "λ=0.0",
    "NoGate\nα=0.5",
    "BAP2\nλ=0.1",
]

n_rows = len(selected)
n_cols = len(columns)

fig_w = n_cols * 2.0
fig_h = n_rows * 2.0 + 0.7

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(fig_w, fig_h),
    dpi=300
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in selected.iterrows():

    father_path = key_to_path(row["key_p_lambda0"])
    mother_path = key_to_path(row["key_m_lambda0"])
    child_path  = key_to_path(row["key_c_lambda0"])

    paths = [
        father_path,
        mother_path,
        child_path,
        row["image_path_lambda0"],
        row["image_path_nogate"],
        row["image_path_bap2"],
    ]

    for c, path in enumerate(paths):
        ax = axes[r, c]
        img = prep_img(path, size=220)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            ax.set_title(columns[c], fontsize=10, pad=6)

    # side label with the triad index
    axes[r, 0].text(
        -0.18, 0.5,
        f"T{int(row['triad_idx'])}",
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=8
    )

plt.subplots_adjust(
    left=0.04,
    right=0.995,
    top=0.93,
    bottom=0.03,
    wspace=0.03,
    hspace=0.06
)

fig.suptitle(
    f"Qualitative comparison of generated descendants at age {AGE_FIG}",
    fontsize=12,
    y=0.985
)

plt.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
plt.show()

print("✓ Figura guardada:")
print(FIG_PNG)
print("✓ Selection CSV:")
print(SEL_CSV)